# EveWatch: Distinguishing Quantum Eavesdroppers from Hardware Noise in BB84
**Track:** Quantum Secure Communication | **Event:** Qiskit Fall Fest 2026 @ MPSTME, NMIMS Mumbai
**Author:** Ayush Trivedi | **GitHub:** https://github.com/Ayush-web28/EveWatch

---

## 1. Executive Summary
This interactive notebook demonstrates the complete **EveWatch** quantum pipeline:
1. Pure Qiskit 2.x implementation of the **BB84 Quantum Key Distribution** protocol.
2. Simulation of an intercept-resend eavesdropper (**Eve**) with configurable interception fraction $f$.
3. Simulation of optical fiber & hardware depolarizing noise ($p_{ch}$).
4. **The Noise vs. Eve Differentiator:** Showing why physical QBER alone is ambiguous, and how statistical sample scaling ($K \ge 600$) resolves False Alarms and Missed Detections.
5. Classical post-processing: Cascade-Lite parity reconciliation, SHA-256 privacy amplification, and One-Time Pad encryption.

In [1]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bb84 import run_bb84, theoretical_qber, binary_entropy, asymptotic_secret_key_rate
from bb84.postprocessing import reconcile_errors_parity, amplify_privacy, encrypt_message_otp, decrypt_message_otp

print('EveWatch modules successfully imported!')

## 2. Baseline BB84: No Eve & No Noise
When transmitting over a noiseless quantum channel without eavesdropping, the Quantum Bit Error Rate (QBER) is strictly 0.00%, and Alice and Bob's sifted keys match identically.

In [2]:
res_clean = run_bb84(num_qubits=500, eve_present=False, channel_noise_prob=0.0, seed=42)
print(f'Transmitted Qubits: {res_clean.num_qubits}')
print(f'Sifted Key Bits: {len(res_clean.sifted_indices)} ({len(res_clean.sifted_indices)/500*100:.1f}%)')
print(f'Sample QBER: {res_clean.qber_sample * 100:.2f}%')
print(f'Security Verdict: {res_clean.verdict}')
print(f'Key Match (Alice == Bob): {np.array_equal(res_clean.alice_key, res_clean.bob_key)}')

## 3. Introducing Eve: Intercept-Resend Attack ($f=1.0$)
Eve intercepts all qubits ($f=1.0$), measures in random bases, and resends. Due to wavefunction collapse when Eve chooses the wrong basis (50% chance), Bob observes an expected **25% error rate** on sifted bits ($Q \approx 0.25 \times f$).

In [3]:
res_eve = run_bb84(num_qubits=1000, eve_present=True, eve_fraction=1.0, channel_noise_prob=0.0, seed=42)
print(f'Full Interception (f=1.0) QBER: {res_eve.qber_overall * 100:.2f}%')
print('Theoretical Expectation: 25.00%')
print(f'Protocol Verdict: {res_eve.verdict}')

## 4. Introducing Hardware & Optical Channel Noise
Depolarizing noise in optical fiber links flips qubit states with probability $p_{ch} / 2$.

In [4]:
res_noise = run_bb84(num_qubits=1000, eve_present=False, channel_noise_prob=0.16, seed=42)
print(f'Channel Noise (p_ch=0.16) QBER: {res_noise.qber_overall * 100:.2f}%')
print('Theoretical Expectation: 8.00%')
print(f'Protocol Verdict: {res_noise.verdict}')

## 5. The Differentiator: Distinguishing Noise from Eve
Notice that **Eve with $f=0.32$** creates ~8% QBER, and **noise with $p_{ch}=0.16$** also creates ~8% QBER!\nNear the 11% cutoff, how do Alice and Bob avoid False Alarms (flagging safe noisy channels) and Missed Detections (letting Eve slip through)?\nBy scaling the sample size $K$, the binomial variance shrinks as $\sigma \propto 1/\sqrt{K}$, eliminating ambiguity.

In [5]:
from bb84.analysis import binomial_detection_probabilities
print(f'{"Sample Size K":15s} | {"False Alarm Rate (Noise Q=9%)":30s} | {"Missed Detection Rate (Eve Q=14%)":32s}')
print('-' * 83)
for k in [50, 150, 300, 600, 1200]:
    far, _ = binomial_detection_probabilities(k, true_qber=0.09, abort_threshold=0.11)
    _, mdr = binomial_detection_probabilities(k, true_qber=0.14, abort_threshold=0.11)
    print(f'{k:15d} | {far*100:29.2f}% | {mdr*100:31.2f}%')

## 6. Classical Post-Processing (Cascade-Lite & Privacy Amplification)
1. **Cascade-Lite:** Parity block checks synchronize keys by correcting channel bit flips.
2. **Privacy Amplification:** SHA-256 compresses key length by $R = 1 - 2 H_2(Q)$ to purge leaked information.

In [6]:
res_post = run_bb84(num_qubits=800, eve_present=False, channel_noise_prob=0.04, seed=123)
print(f'Initial Sample QBER: {res_post.qber_sample * 100:.2f}%')
rec_a, rec_b, corrections = reconcile_errors_parity(res_post.alice_key, res_post.bob_key, block_size=8)
print(f'Parity bisection corrections applied: {corrections}')
print(f'Reconciled Keys Synchronized: {np.array_equal(rec_a, rec_b)}')

secret_key = amplify_privacy(rec_a, estimated_qber=res_post.qber_sample)
print(f'Final Distilled Secret Key: {len(secret_key)} bits')
print(f'Secret Key Preview: {"".join(str(b) for b in secret_key[:32])}...')

## 7. One-Time Pad Encryption Demonstration
Finally, Alice encrypts a sensitive transmission using One-Time Pad (XOR) with the distilled secret key.

In [7]:
message = 'Quantum Secure Communication @ Qiskit Fall Fest 2026!'
hex_cipher, bin_cipher = encrypt_message_otp(message, secret_key)
print('Alice Plaintext:', message)
print('Encrypted Ciphertext (Hex):', hex_cipher)
decrypted = decrypt_message_otp(hex_cipher, secret_key)
print('Bob Decrypted Plaintext:', decrypted)
assert decrypted == message, 'Decryption mismatch!'
print('Verification Passed: 100% information-theoretic security confirmed.')